In [1]:
import kagglehub
path = kagglehub.dataset_download("uom190346a/sleep-health-and-lifestyle-dataset")

Using Colab cache for faster access to the 'sleep-health-and-lifestyle-dataset' dataset.


In [2]:
import os
import pandas as pd

# List files in the downloaded dataset path
print("Downloaded files:", os.listdir(path))

# Assuming there is a CSV file, find its path and load it
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if csv_files:
    file_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(file_path)
    print(f"Loaded {csv_files[0]} successfully!")
    display(df.head())
    display(df.info())
else:
    print("No CSV file found in the dataset folder.")

Downloaded files: ['Sleep_health_and_lifestyle_dataset.csv']
Loaded Sleep_health_and_lifestyle_dataset.csv successfully!


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Person ID                374 non-null    int64  
 1   Gender                   374 non-null    object 
 2   Age                      374 non-null    int64  
 3   Occupation               374 non-null    object 
 4   Sleep Duration           374 non-null    float64
 5   Quality of Sleep         374 non-null    int64  
 6   Physical Activity Level  374 non-null    int64  
 7   Stress Level             374 non-null    int64  
 8   BMI Category             374 non-null    object 
 9   Blood Pressure           374 non-null    object 
 10  Heart Rate               374 non-null    int64  
 11  Daily Steps              374 non-null    int64  
 12  Sleep Disorder           155 non-null    object 
dtypes: float64(1), int64(7), object(5)
memory usage: 38.1+ KB


None

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

# 1. Preprocessing the data
data = df.copy()

# Drop identifier column
data = data.drop(columns=['Person ID'])

# Fill NaN in Sleep Disorder with 'None'
data['Sleep Disorder'] = data['Sleep Disorder'].fillna('None')

# Split Blood Pressure into Systolic and Diastolic
data[['Systolic_BP', 'Diastolic_BP']] = data['Blood Pressure'].str.split('/', expand=True).astype(float)
data = data.drop(columns=['Blood Pressure'])

# Define feature columns and target
target_col = 'Sleep Disorder'
X = data.drop(columns=[target_col])
y = data[target_col]

# Define categoricals and numericals
categorical_cols = ['Gender', 'Occupation', 'BMI Category']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Normalize target labels
target_encoder = OneHotEncoder(sparse_output=False)
y_encoded = target_encoder.fit_transform(y.values.reshape(-1, 1))

# Preprocess features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

X_processed = preprocessor.fit_transform(X)

# Split to train and test
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_encoded, test_size=0.2, random_state=42, stratify=y)

# 2. Build the Deep Learning Model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    BatchNormalization(),

    Dense(y_encoded.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 3. Train the Model
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

# 4. Evaluate the Model
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}\n")

# Detail classification report
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

print(classification_report(
    y_test_classes,
    y_pred_classes,
    target_names=target_encoder.categories_[0]
))

Epoch 1/100


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.5316 - loss: 1.2057 - val_accuracy: 0.7000 - val_loss: 0.8989
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7435 - loss: 0.7182 - val_accuracy: 0.9000 - val_loss: 0.7613
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8178 - loss: 0.5494 - val_accuracy: 0.9333 - val_loss: 0.6600
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8290 - loss: 0.5690 - val_accuracy: 0.9333 - val_loss: 0.6009
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8476 - loss: 0.5587 - val_accuracy: 0.9000 - val_loss: 0.5490
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8736 - loss: 0.5039 - val_accuracy: 0.9333 - val_loss: 0.5095
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8513 - loss: 0.4884 - val_accuracy: 0.9333 - val_loss: 0.4664
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8773 - loss: 0.4728 - val_accuracy: 0.9333 - val_loss: 0.

In [4]:
import pickle

# Save the model and preprocessing pipeline so the Streamlit app can load them
model.save('sleep_model.keras')

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

with open('target_encoder.pkl', 'wb') as f:
    pickle.dump(target_encoder, f)

print("Model and preprocessor saved successfully!")

Model and preprocessor saved successfully!


In [5]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle

# Page config
st.set_page_config(
    page_title="Sleep Health Predictor",
    page_icon="💤",
    layout="wide"
)

# Title and description
st.title("💤 Sleep Health & Lifestyle Predictor")
st.markdown("### Discover your sleep profile using Advanced Deep Learning 🚀")

# Load artifacts
@st.cache_resource
def load_artifacts():
    model = tf.keras.models.load_model('sleep_model.keras')
    with open('preprocessor.pkl', 'rb') as f:
        preprocessor = pickle.load(f)
    with open('target_encoder.pkl', 'rb') as f:
        target_encoder = pickle.load(f)
    return model, preprocessor, target_encoder

try:
    model, preprocessor, target_encoder = load_artifacts()
except Exception as e:
    st.error(f"Error loading model artifacts: {e}")

# Sidebar Inputs
st.sidebar.header("👤 User Profile & Habits")

gender = st.sidebar.selectbox("👩‍💼 Gender", ["Male", "Female"])
age = st.sidebar.slider("🎂 Age", 10, 100, 35)

occupation_list = [
    'Software Engineer', 'Doctor', 'Sales Representative', 'Teacher',
    'Nurse', 'Engineer', 'Accountant', 'Scientist', 'Lawyer',
    'Manager', 'Salesperson'
]
occupation = st.sidebar.selectbox("💼 Occupation", occupation_list)
sleep_duration = st.sidebar.slider("⏱️ Sleep Duration (Hours)", 3.0, 10.0, 7.0, 0.1)
quality_sleep = st.sidebar.slider("⭐ Quality of Sleep (1-10)", 1, 10, 6)
activity_level = st.sidebar.slider("🏃‍♂️ Physical Activity Level (Minutes/Day)", 10, 120, 60)
stress_level = st.sidebar.slider("🤯 Stress Level (1-10)", 1, 10, 5)
bmi_category = st.sidebar.selectbox("⚖️ BMI Category", ["Normal", "Overweight", "Obese"])
heart_rate = st.sidebar.slider("❤️ Heart Rate (bpm)", 50, 120, 72)
daily_steps = st.sidebar.slider("👣 Daily Steps", 1000, 20000, 6000, 100)

systolic_bp = st.sidebar.slider("📈 Systolic Blood Pressure (mmHg)", 90, 180, 120)
diastolic_bp = st.sidebar.slider("📉 Diastolic Blood Pressure (mmHg)", 60, 120, 80)

# Main panel layout
col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("🔬 Analyzed Profile Features")

    # Display inputs nicely
    profile_df = pd.DataFrame([{
        "Gender": gender, "Age": age, "Occupation": occupation,
        "Sleep Duration": sleep_duration, "Quality of Sleep": quality_sleep,
        "Physical Activity Level": activity_level, "Stress Level": stress_level,
        "BMI Category": bmi_category, "Heart Rate": heart_rate,
        "Daily Steps": daily_steps, "Systolic_BP": systolic_bp, "Diastolic_BP": diastolic_bp
    }])
    st.table(profile_df.T.rename(columns={0: "Value"}))

with col2:
    st.subheader("🔮 AI Diagnostic Prediction")

    if st.button("Predict Sleep Status ✨", use_container_width=True):
        # Process the input data
        processed_input = preprocessor.transform(profile_df)

        # Predict probability
        prediction_prob = model.predict(processed_input)
        pred_class_idx = np.argmax(prediction_prob, axis=1)[0]

        # Decoded labels
        classes = target_encoder.categories_[0]
        prediction_label = classes[pred_class_idx]

        st.success(f"### Prediction: **{prediction_label}**")

        # Display confidence
        for cls, prob in zip(classes, prediction_prob[0]):
            st.metric(label=f"📊 Probability for: {cls}", value=f"{prob*100:.2f}%")

Writing app.py


In [7]:
!pip install --ignore-installed blinker
!pip install streamlit

In [8]:
!pip install streamlit pyngrok

In [9]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [10]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [11]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run app.py &




2026-11-14 09:42:37.689 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.73.115.163:8501

I0000 00:00:1794649369.796682    3004 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794649372.559085    3004 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1794649375.529892    3004 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-11-14 09:42:56.093 Serialization of dataframe to Arrow table was unsuccessful. Applying automatic fixes for column types to make the dataframe Arrow-compatible.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/streamlit/dataframe_util.py", line 967, in convert_pandas_df_to_arrow_table
    return pa.Table.from_pandas(df, preserve_i

  Stopping...
